<hr>
<center> Control Systems - Ashesi University - January 2026 <br>
<b> Take-home Project for Control Systems </b> <br>
Jonas Matt, Verena Häberle, Prof. Florian Dörfler <br>
Automatic Control Laboratory, ETH Zurich </center>
<hr>

In this notebook, you will find different code fragments to complete the take-home project. <b>

<!-- <blockquote> -->
<b>Activity:</b> Execute the code cells below so that the necessary libraries get imported.
<!-- </blockquote> -->

In [ ]:
# if you run code on google colab these lines will be necessary
!apt-get install gfortran cmake --fix-missing
!apt-get install libblas-dev liblapack-dev

import os
os.environ['BLA_VENDOR']="Generic"
!echo Using $$BLA_VENDOR BLAS implementation

!mkdir figure

!pip install -v slycot
!pip install control
!pip install matplotlib
!pip install ipympl

from google.colab import output
output.enable_custom_widget_manager()

import numpy as np # the standard library for numerics, vectors, matrices
import control as ct # the standard library for basic operations for analysis and design of feedback control systems
import matplotlib.pyplot as plt # a comprehensive library for creating static, animated, and interactive visualizations
import sympy as sp # the standard library for symbolic mathematical computations

This example shows you how to use the *sympy* package to compute the jacobian of the function <br>
\begin{equation}
f(s) = \begin{bmatrix}
s_1^2 + s_2 \\ s_1^3 - s_2^2
\end{bmatrix}
\end{equation}
with respect to the variable <br>
\begin{equation}
s=\begin{bmatrix}
s_1\\ s_2
\end{bmatrix}.
\end{equation}
The Jacobian is then evaluated at $s_1 =1$. This will be useful for task T1-i).

In [ ]:
s1, s2 = sp.symbols('s1, s2')
s_vec = sp.Array([[s1],[s2]])
f = sp.Matrix([s1**2 + s2, s1**3 - s2**2])
Delta_f = f.jacobian(s_vec)
print(Delta_f)
s_param = {"s1": 1}
Delta_s_param = Delta_f.subs(s_param)
print(Delta_s_param)

<b> Code fragment for T1 linearization </b>

In [ ]:
# define symbolic variables
x, y, z, xdot, ydot, zdot, phi, theta, psi, p, q, r = sp.symbols('x, y, z, xdot, ydot, zdot, phi, theta, psi, p, q, r')
u1,u2,u3,u4, = sp.symbols('u1, u2, u3, u4')
m, g, Ix, Iy, Iz, kx, ky, kz, kp, kq, kr = sp.symbols('m, g, Ix, Iy, Iz, kx, ky, kz, kp, kq, kr')
xd, yd, zd, psid = sp.symbols('xd, yd, zd, psid')

# define symbolic state, input & equilibrium vector
states = sp.Array([[x], [y], [z], [xdot], [ydot], [zdot], [phi], [theta], [psi], [p], [q], [r]])
inputs = sp.Array([[u1], [u2], [u3], [u4]])
equilibrium = sp.Array([[xd], [yd], [zd], [0], [0], [0], [0], [0], [psid], [0], [0], [0]])

# nonlinear model
xddot = (1/m) * (u1 * (sp.sin(psi) * sp.sin(phi) + sp.cos(psi) * sp.cos(phi) * sp.sin(theta)) - kx * xdot)
yddot = (1/m) * (u1 * (sp.cos(psi) * sp.sin(phi) - sp.sin(psi) * sp.cos(phi) * sp.sin(theta)) - ky * ydot)
zddot = (1/m) * (u1 * sp.cos(phi) * sp.cos(theta) - m * g - kz * zdot)
phiddot = (1/Ix) * (u2 - kp * p)
thetaddot = (1/Iy) * (u3 - kq * q)
psiddot = (1/Iz) * (u4 - kr * r)
drone = sp.Matrix([xdot, yddot, zddot, xddot, yddot, zddot, phi, theta, psi, p, q, r])

# compute symbolic state-space matrices
eq_param = {"x":xd,"y":yd,"z":zd, "xdot":0, "ydot":0, "zdot":0, "phi":0, "theta":0, "psi":psid, "p":0, "q":0, "r":0}
Asym = drone.jacobian(states)
Asym_eq = Asym.subs(eq_param)
Bsym = drone.jacobian(inputs)
Bsym_eq = Bsym.subs(eq_param)

# compute numeric state-space matrices with dtype float
param = {"m":0.03, "g":9.81, "kx":4.5e-3, "ky":4.5e-3, "kz":4.5e-3, "Ix":1.5e-5, "Iy":1.5e-5, "Iz":3e-5, "kp":4.5e-4, "kq":4.5e-4, "kr":4.5e-4}
A = np.array(Asym_eq.subs(param),dtype = float)
B = np.array(Bsym_eq.subs(param),dtype = float)
C = np.eye(12)
C = C[[0,1,2,6],:] # select only x, y, z, phi

# print system matrices A, B
np.set_printoptions(precision=4,suppress=True,linewidth=np.inf)
print('The system matrix A is \n',A)
print('The system matrix B is \n',B)

<br>

<br>

<b> Code fragment for T2 stability analysis </b>

In [ ]:
# compute eigenvalues
eigenvalues = np.linalg.eigvals(A)
print("Eigenvalues:", eigenvalues)

# compute Jordan form
# Hint: To compute the Jordan form of the matrix M, use the command 'M.jordan_form()', which can only be applied if M is a Sympy matrix.
A_sympy = sp.Matrix(A)
P, J = A_sympy.jordan_form()
print("Jordan form:", J)
print("Transformation matrix P:", P)

<br>

<b> Code fragment for T3 forced responses </b>

In [ ]:
# extracting the rolling dynamics
A_rolling = A[[6, 9], :][:, [6, 9]]  # rows 6,9 and cols 6,9
B_rolling = B[[6, 9], 1:2]  # rows 6,9 and col 1 (u2)
C_rolling = np.array([[1, 0]])  # output is phi
D_rolling = np.array([[0]])
sys_rolling = ct.StateSpace(A_rolling, B_rolling, C_rolling, D_rolling)

# plot impulse response
t_impulse, y_impulse = ct.impulse_response(sys_rolling, T=np.linspace(0, 10, 1000), U=1e-5*np.ones(1000))

# plot step response
t_step, y_step = ct.step_response(sys_rolling, T=np.linspace(0, 10, 1000), U=1e-5*np.ones(1000))

# plot sinusoidal response
t_sin = np.linspace(0, 10, 1000)
u_sin = 1e-5 * np.sin(t_sin)
t_sin, y_sin = ct.forced_response(sys_rolling, T=t_sin, U=u_sin)

<br>

<b> Code fragment for T4 state-feedback control </b>

In [ ]:
# set parameters
m = 0.03
kx = 0.015
kq = 4.5e-4
Iy = 1.5e-5
g = 9.81

# set system matrices
A_x = np.array([[0, 1, 0, 0],
                [0, -kx/m, g, 0],
                [0, 0, 0, 1],
                [0, 0, 0, -kq/Iy]])

B_x = np.array([[0], [0], [0], [1/Iy]])

C_x = np.array([[1, 0, 0, 0]])

D_x = np.array([[0]])

# analyze stability
eigenvalues_Ax = np.linalg.eigvals(A_x)
print("Eigenvalues of A_x:", eigenvalues_Ax)

# analyze controllability
ctrb_matrix = ct.ctrb(A_x, B_x)
rank_ctrb = np.linalg.matrix_rank(ctrb_matrix)
print("Controllability matrix rank:", rank_ctrb, "System dimension:", A_x.shape[0])
if rank_ctrb == A_x.shape[0]:
    print("System is controllable")
else:
    print("System is not controllable")

In [ ]:
# pole placement
desired_poles = np.array([-100, -50, -10, -10])
K = ct.place(A_x, B_x, desired_poles)
print("State feedback gain K:", K)

# closed-loop system
A_cl = A_x - B_x @ K
eigenvalues_cl = np.linalg.eigvals(A_cl)
print("Closed-loop eigenvalues:", eigenvalues_cl)

<b> Code fragment for T5 augmented state-feedback control </b>

In [ ]:
# integral control
# augment the system with integral state
A_aug = np.block([[A_x, np.zeros((4, 1))],
                  [-C_x, np.zeros((1, 1))]])
B_aug = np.block([[B_x], [np.zeros((1, 1))]])

# design state feedback for augmented system
desired_poles_aug = np.array([-100, -50, -10, -10, -5])
K_aug = ct.place(A_aug, B_aug, desired_poles_aug)
print("Augmented state feedback gain K_aug:", K_aug)

# simulate with reference r = 1
A_cl_aug = A_aug - B_aug @ K_aug
sys_aug = ct.StateSpace(A_cl_aug, B_aug, np.eye(5), np.zeros((5, 1)))

T_ref = np.linspace(0, 10, 1000)
t_pos, x_pos = ct.initial_response(sys_aug, T=T_ref, X0=np.array([0, 0, 0, 0, 0]))
x_pos = x_pos[0, :]  # extract just the x position (first state)

plt.figure()
plt.plot(T_ref,np.ones(T_ref.shape),linewidth=2)
plt.plot(t_pos,x_pos,linewidth=2)
plt.legend(["Reference", "Trajectory"], loc ="lower right",fontsize=14)
plt.xlabel("Time (s)",fontsize=14)
plt.ylabel("Position x (m)",fontsize=14)
plt.title("Trajectory",fontsize=16)
plt.grid()
#plt.savefig('figure/traj_x.png',bbox_inches='tight')
plt.show()

In [ ]:
# feedforward & feedback control
# compute feedforward gain to track step reference r = 1
# For steady-state tracking: x_ss = 1
# We use: u_ff = -C_x @ A_x^{-1} @ B_x @ u_ff, solve for scaling
# Simpler: N = -(C @ (A - B K)^{-1} @ B)^{-1}

# use the state feedback from T4 (without augmentation)
K_t4 = ct.place(A_x, B_x, np.array([-100, -50, -10, -10]))
A_cl_t4 = A_x - B_x @ K_t4

# compute feedforward gain for step tracking
# N = -1 / (C @ inv(A_cl) @ B)
N = -1.0 / (C_x @ np.linalg.inv(A_cl_t4) @ B_x)[0, 0]
print("Feedforward gain N:", N)

# closed-loop system with feedforward: u = N*r - K*x
A_ffcl = A_x - B_x @ K_t4
sys_ffcl = ct.StateSpace(A_ffcl, N * B_x, C_x, D_x)

T_ref = np.linspace(0, 10, 1000)
t_pos, x_pos = ct.step_response(sys_ffcl, T=T_ref)

plt.figure()
plt.plot(T_ref, np.ones_like(T_ref), linewidth=2)        # reference r = 1
plt.plot(t_pos, x_pos, linewidth=2)                      # output y = x position
plt.legend(["Reference", "Trajectory"], loc="lower right", fontsize=14)
plt.xlabel("Time (s)", fontsize=14)
plt.ylabel("Position x (m)", fontsize=14)
plt.title("Trajectory with Prefiltered State Feedback", fontsize=16)
plt.grid()
#plt.savefig('figure/traj_x.png', bbox_inches='tight')
plt.show()

<br>

<b> Code fragments for T6 observer-based control </b>

In [ ]:
# set model parameters
m = 0.03
kx = 4.5e-3 # new value for kx!
g = 9.81
Iy = 1.5e-5
kq = 4.5e-4

# set system matrices
A_x = np.array([[0, 1, 0, 0],
                [0, -kx/m, g, 0],
                [0, 0, 0, 1],
                [0, 0, 0, -kq/Iy]])

B_x = np.array([[0], [0], [0], [1/Iy]])

C_x = np.array([[1, 0, 0, 0]])

D_x = np.array([[0]])

# design of the state-feedback controller
K_poles = np.array([-0.3, -0.4, -10, -20])
K = ct.place(A_x, B_x, K_poles)
print("State feedback gain K:", K)

# design of the observer
L_poles = np.array([-23, -26, -27, -28])
L = ct.place(A_x.T, C_x.T, L_poles).T
print("Observer gain L:", L)

# augmented system: [x_actual; x_estimated]
A_aug = np.block([[A_x - B_x @ K, B_x @ K],
                  [np.zeros((4, 4)), A_x - L @ C_x]])
B_aug = np.block([[np.zeros((4, 1))],
                  [L]])
C_aug = np.block([[C_x, -C_x]])  # output: x - x_hat (estimation error)

sys_aug = ct.StateSpace(A_aug, B_aug, C_aug, np.array([[0]]))

# plot the closed-loop trajectories
plt.figure()
t_ref = np.linspace(0, 10, 1000)
X0 = np.array([-2, 0, 0, 0, 0, 0, 0, 0])  # initial conditions: [actual_x; estimated_x]
t_aug, y_aug = ct.initial_response(sys_aug, T=t_ref, X0=X0)

plt.plot(t_ref, y_aug[0, :], linewidth=2, label='System output x')
# Note: y_aug contains estimation error already
plt.legend(['System output', 'Estimation error'], loc="lower right", fontsize=14)
plt.xlabel('Time (s)', fontsize=14)
plt.ylabel('Position (m)', fontsize=14)
plt.title('Closed-Loop Trajectories', fontsize=16)
plt.grid()
#plt.savefig('figure/closed_loop_traj.png',bbox_inches='tight')
plt.show()